# CAS Exam 5: Frequency-Severity Techniques and Disposal Rate Method

This notebook demonstrates:
1. Frequency-Severity Technique #1
2. Frequency-Severity Technique #2
3. Disposal Rate Method

Data used from `chainladder/utils/data`:
- `friedland_xyz_freq_sev.csv`
- `friedland_xyz_disp.csv`
- `xyz.csv` (premium used as an exposure proxy for Technique #2)


## Formula Sheet Reference

### Frequency-Severity Technique #1: Develop Counts and Severity Separately

$$\text{Ultimate Claims}_w = \text{Ultimate Counts}_w \times \text{Ultimate Severity}_w$$

**Where:**
- Ultimate Counts_w = Latest Closed Counts_w × CDF_counts(d → ultimate)
- Ultimate Severity_w = Latest Reported Severity_w × CDF_severity(d → ultimate)
- IBNR = Ultimate Claims_w − Latest Reported Claims_w

**Data requirements:** Cumulative closed claim count triangle + reported severity triangle + latest reported claims for IBNR calculation.

### Frequency-Severity Technique #2: Trend-Projected Frequency and Severity

$$\text{Frequency}_w = \frac{\text{Ultimate Counts}_w}{\text{Exposure}_w}$$

For mature AYs (development age ≥ 84 months), fit a log-linear trend:

$$\ln(\text{Frequency}_w) = \hat{a} + \hat{b} \times w \implies \text{Projected Frequency}_w = e^{\hat{a} + \hat{b} w}$$

$$\ln(\text{Severity}_w) = \hat{c} + \hat{d} \times w \implies \text{Projected Severity}_w = e^{\hat{c} + \hat{d} w}$$

$$\text{Ultimate Claims}_w = \text{Exposure}_w \times \text{Projected Frequency}_w \times \text{Projected Severity}_w$$

**Advantage over Technique #1:** Immature AY projections are driven by trend extrapolation, not by potentially unreliable development from sparse early data.

### Disposal Rate Method

$$\text{DR}(w, d) = \frac{\text{Cumulative Closed Counts}_{w,d}}{\text{Ultimate Closed Counts}_w}$$

Rearranged to solve for ultimate:

$$\text{Ultimate Closed Counts}_w = \frac{\text{Cumulative Closed}_{w,d}}{\text{DR}(d)}$$

**Projection:**

$$\text{Projected Incremental Closed}_{d \to d+1} = \text{Ultimate Closed}_w \times \left(\text{DR}_{d+1} - \text{DR}_d\right)$$

$$\text{Projected Incremental Paid}_{d \to d+1} = \text{Projected Inc. Closed} \times \text{Incremental Severity}_{d \to d+1}$$

$$\text{Ultimate Paid}_w = \text{Latest Paid}_w + \sum_{\text{future ages}} \text{Projected Incremental Paid}$$

### When to Use Each Method

| Method | Best Conditions | Avoid When |
|---|---|---|
| Freq-Sev Technique #1 | Count and severity triangles are credible and stable; closed-count development is mature | Severity is volatile at late ages; count triangle is sparse |
| Freq-Sev Technique #2 | Immature AYs make Technique #1 unreliable; exposure base is available; trend data is credible | Frequency/severity trends are unstable; fewer than 4–5 mature AYs for calibration |
| Disposal Rate | Closed-count emergence is credible; incremental severity is stable by development age | Long-tailed lines with very few counts at late ages; incremental severity is erratic |

### Key Assumptions

1. Count triangle follows a stable cumulative development pattern (all three methods)
2. Severity triangle reflects stable cost levels (Freq-Sev Technique #1)
3. Disposal rates are consistent across origin years at the same development age (Disposal Rate)
4. Incremental severity by development age is stable across origin years (Disposal Rate)
5. Frequency and severity trends are log-linear and stable in the calibration window (Technique #2)

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import chainladder as cl

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import triangle_to_frame

DATA_DIR = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data'
freq_sev_df = pd.read_csv(DATA_DIR / 'friedland_xyz_freq_sev.csv')
disp_df = pd.read_csv(DATA_DIR / 'friedland_xyz_disp.csv')
xyz_df = pd.read_csv(DATA_DIR / 'xyz.csv')

freq_sev_triangle = cl.Triangle(
    freq_sev_df,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Closed Claim Counts', 'Reported Claim Counts', 'Reported Claims', 'Reported Severities'],
    cumulative=True,
)

closed_count_triangle = freq_sev_triangle['Closed Claim Counts']
reported_claims_triangle = freq_sev_triangle['Reported Claims']
reported_severity_triangle = freq_sev_triangle['Reported Severities']

{'freq_sev_triangle_shape': freq_sev_triangle.shape, 'valuation_date': str(freq_sev_triangle.valuation_date)}


In [ ]:
latest_counts = closed_count_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_claims = reported_claims_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_severity = reported_severity_triangle.latest_diagonal.to_frame().iloc[:, 0]

closed_long = triangle_to_frame(closed_count_triangle, origin_as_datetime=False).reset_index()
closed_matrix = closed_long.pivot(index='origin', columns='development', values='Closed Claim Counts').sort_index().sort_index(axis=1)
latest_age = closed_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_age.index = pd.to_datetime(latest_age.index.astype(str) + "-01-01")
latest_age = latest_age.reindex(latest_counts.index)

triangle_snapshot = pd.DataFrame(
    {
        'LatestClosedCounts': latest_counts.values,
        'LatestReportedClaims': latest_claims.values,
        'LatestReportedSeverity': latest_severity.values,
        'LatestMaturityAge': latest_age.values,
    },
    index=latest_counts.index.year,
)
triangle_snapshot.index.name = 'AccidentYear'
triangle_snapshot


## Pre-Analysis: Count Triangle Quality Check

Before applying any frequency-severity or disposal rate method, verify that the closed count triangle has stable development. The `count_triangle_diagnostics` function produces the standard link ratio exhibit for the count triangle plus the percent-closed-by-age series derived from the volume-weighted LDF.

In [ ]:
from reservingengine.reserving import count_triangle_diagnostics

cd = count_triangle_diagnostics(closed_count_triangle)

print("Count triangle link ratio exhibit:")
display(cd['link_ratio_table'].style.format("{:.4f}", na_rep="---"))

print("\nPercent closed by development age (1 − 1/VolWtd LDF):")
cd['percent_closed_by_age'].to_frame('percent_closed').style.format("{:.1%}")

**EXAM RED FLAG —** If `percent_closed` at the latest development age is below 90%, there is significant remaining count development. The tail of the disposal rate method depends entirely on how you handle those late-age claims — a poor tail severity assumption can be a larger source of reserve error than the bulk of the projection.

**EXAM RED FLAG —** If `percent_closed` reaches ≥ 98% by the latest development age, the count tail is nearly complete and both the disposal rate method and Frequency-Severity Technique #1 are well-supported by the data.

**Freq-Sev convergence check:** If reported severity link ratios are near 1.000 from early development ages (severity appears fully developed quickly), but paid claims continue to develop (more claims are closing at later ages), then counts are driving ALL remaining development. In this case, Frequency-Severity Technique #1 and the Disposal Rate method will produce similar results — a useful sanity check when both are available.

See `exam5_diagnostics.ipynb` Section 9 for full count triangle diagnostic walkthrough.

## Frequency-Severity Technique #1

Approach:
1. Apply development method to cumulative closed claim counts.
2. Apply development method to reported severities.
3. Multiply projected ultimate counts x projected ultimate severity.
4. Compute IBNR as projected ultimate claims minus latest reported claims.

Assumption emphasis:
- Count and severity patterns observed to date remain representative for future development.

In [ ]:
count_dev = cl.Development(average='volume', n_periods=-1).fit_transform(closed_count_triangle)
severity_dev = cl.Development(average='volume', n_periods=-1).fit_transform(reported_severity_triangle)

count_model = cl.Chainladder().fit(count_dev)
severity_model = cl.Chainladder().fit(severity_dev)

ultimate_counts_t1 = count_model.ultimate_.to_frame().iloc[:, 0]
ultimate_severity_t1 = severity_model.ultimate_.to_frame().iloc[:, 0]
ultimate_claims_t1 = ultimate_counts_t1 * ultimate_severity_t1
ibnr_t1 = ultimate_claims_t1 - latest_claims

tech1_ay = pd.DataFrame(
    {
        'UltimateCounts_T1': ultimate_counts_t1.values,
        'UltimateSeverity_T1': ultimate_severity_t1.values,
        'UltimateClaims_T1': ultimate_claims_t1.values,
        'LatestReportedClaims': latest_claims.values,
        'IBNR_T1': ibnr_t1.values,
    },
    index=ultimate_counts_t1.index.year,
)
tech1_ay.index.name = 'AccidentYear'
tech1_ay.loc['Total'] = tech1_ay.sum()
tech1_ay


## Frequency-Severity Technique #2

Approach:
1. Start from projected ultimate counts.
2. Convert to frequency by dividing by exposure.
3. Select frequency and severity using trend analysis (especially for immature years).
4. Project ultimate claims as: Exposure x Selected Frequency x Selected Severity.

In this demo, premium from `xyz.csv` is used as an exposure proxy.


In [ ]:
exposure_proxy = xyz_df.groupby('AccidentYear')['Premium'].max().sort_index()
exposure_proxy.index = pd.to_datetime(exposure_proxy.index.astype(str) + "-01-01")
exposure_proxy = exposure_proxy.reindex(ultimate_counts_t1.index)

frequency_from_t1 = ultimate_counts_t1 / exposure_proxy
severity_from_t1 = ultimate_severity_t1.copy()

mature_mask = latest_age >= 84
ay_numeric = pd.Series(ultimate_counts_t1.index.year, index=ultimate_counts_t1.index, dtype=float)

freq_fit_mask = mature_mask & frequency_from_t1.notna() & (frequency_from_t1 > 0)
sev_fit_mask = mature_mask & severity_from_t1.notna() & (severity_from_t1 > 0)

if int(freq_fit_mask.sum()) >= 2:
    freq_slope, freq_intercept = np.polyfit(
        ay_numeric[freq_fit_mask].to_numpy(dtype=float),
        np.log(frequency_from_t1[freq_fit_mask].to_numpy(dtype=float)),
        1,
    )
    selected_frequency_t2 = np.exp(freq_intercept + freq_slope * ay_numeric.to_numpy(dtype=float))
else:
    selected_frequency_t2 = np.repeat(float(frequency_from_t1.mean()), len(ay_numeric))

if int(sev_fit_mask.sum()) >= 2:
    sev_slope, sev_intercept = np.polyfit(
        ay_numeric[sev_fit_mask].to_numpy(dtype=float),
        np.log(severity_from_t1[sev_fit_mask].to_numpy(dtype=float)),
        1,
    )
    selected_severity_t2 = np.exp(sev_intercept + sev_slope * ay_numeric.to_numpy(dtype=float))
else:
    selected_severity_t2 = np.repeat(float(severity_from_t1.mean()), len(ay_numeric))

selected_frequency_t2 = pd.Series(selected_frequency_t2, index=ultimate_counts_t1.index)
selected_severity_t2 = pd.Series(selected_severity_t2, index=ultimate_counts_t1.index)

ultimate_claims_t2 = exposure_proxy * selected_frequency_t2 * selected_severity_t2
ibnr_t2 = ultimate_claims_t2 - latest_claims

tech2_ay = pd.DataFrame(
    {
        'ExposureProxy': exposure_proxy.values,
        'SelectedFrequency_T2': selected_frequency_t2.values,
        'SelectedSeverity_T2': selected_severity_t2.values,
        'UltimateClaims_T2': ultimate_claims_t2.values,
        'LatestReportedClaims': latest_claims.values,
        'IBNR_T2': ibnr_t2.values,
    },
    index=ultimate_claims_t2.index.year,
)
tech2_ay.index.name = 'AccidentYear'
tech2_ay.loc['Total'] = tech2_ay.sum()
tech2_ay


## Frequency-Severity Method Comparison and Red-Flag Notes

**Advantages of Freq-Sev over pure development (chain ladder):**
- More stable for immature AYs — counts develop more smoothly than dollar amounts
- Provides insight into separate frequency and severity drivers
- Allows explicit incorporation of inflation (Technique #2 trend)
- Reduces leverage of chain ladder on very immature accident years

**Disadvantages:**
- Higher data requirements (count triangles, exposure data)
- Frequency and severity trends are additional assumptions with their own uncertainty
- Sensitive to incorrect trend selection (especially for Technique #2)

---

**EXAM RED FLAG —** In Technique #2, frequency and severity trends are fit independently from mature AYs. If fewer than 4–5 accident years are mature enough to calibrate the trend (development age ≥ 84 months), the slope estimate carries high uncertainty. On the exam, if n_mature_origins is small, treat Technique #2 projections for immature AYs as low credibility.

**EXAM RED FLAG —** The frequency and severity trends in Technique #2 are fit as independent log-linear regressions. If severity trend is driven by inflation that also affects frequency (e.g., more complex claims are both more numerous AND more expensive), the interaction is understated — the combined ultimate may be understated as a result.

**EXAM RED FLAG —** The Disposal Rate method assumes selected disposal rates by age are consistent across origin years. If settlement practices changed materially mid-triangle (e.g., a faster settlement strategy was adopted), the selected DR by age will blend inconsistent origin years — this is the same homogeneity problem that motivates Berquist-Sherman adjustment for paid triangles.

In [ ]:
final_summary = pd.DataFrame(
    {
        'Method': [
            'Freq-Sev Technique #1',
            'Freq-Sev Technique #2',
            'Disposal Rate Method',
        ],
        'TotalUltimateClaims': [
            float(ultimate_claims_t1.sum()),
            float(ultimate_claims_t2.sum()),
            float(disposal_projection.loc['Total', 'ProjectedUltimatePaidClaims']),
        ],
        'TotalIBNR': [
            float(ibnr_t1.sum()),
            float(ibnr_t2.sum()),
            float(disposal_projection.loc['Total', 'IBNR_DisposalMethod']),
        ],
    }
)

impact_table = pd.DataFrame(
    [
        [
            'Speedup in settlement rate',
            'Closed counts develop faster — count LDFs are lower than historical; ultimate count estimate drops; IBNR may be understated if not adjusted',
            'Disposal rates shift upward at early ages — selected DRs need to be from a consistent-settlement-rate period',
            'Frequency increases (more claims close faster) — Technique #2 captures this via trend if the change is gradual',
        ],
        [
            'Increase in claim severity (inflation)',
            'Severity triangle develops upward — severity LDFs increase; ultimate claims increase proportionally',
            'Incremental severity by development age increases — tail severity assumption increases; overall ultimate paid increases',
            'Severity trend slope captures the inflation if it falls within the calibration window; projected severity for immature AYs increases',
        ],
        [
            'Change in product mix (more complex claims)',
            'Severity increases; count closure rates may slow — both effects require re-evaluation of count and severity LDFs',
            'Disposal rates may shift lower (slower closure of complex claims)',
            'Frequency and severity both affected — may need to re-segment if mix change is material',
        ],
        [
            'Exposure growth',
            'Ultimate counts increase proportionally; ultimate claims increase; no explicit adjustment needed if severity is stable',
            'Not directly reflected in disposal rate projection (exposure is not explicit in the formula)',
            'Exposure enters directly in Technique #2 formula — ultimate claims grow proportionally with exposure',
        ],
        [
            'Average accident date shifts forward',
            'Count and severity CDFs understated at observed ages — less development observed in triangle window than would occur at later ages',
            'Disposal rates understate ultimate at each observed age',
            'Mature AY trend calibration may be optimistic for immature AYs if accident date shift is material',
        ],
        [
            'Sparse late-age data (long-tail line)',
            'Severity LDFs at late ages have low credibility — smoothing or external benchmarks needed',
            'Incremental severity at late ages is volatile — tail severity assumption carries high weight; key judgmental assumption',
            'Technique #2 avoids explicit late-age data by using trend extrapolation — may be preferred for long-tail lines',
        ],
    ],
    columns=['Change in Environment', 'Impact on Freq-Sev Techniques', 'Impact on Disposal Rate Method', 'Notes for Technique #2'],
)

print("Method comparison summary:")
display(final_summary)

print("\nEnvironment impact table:")
impact_table

In [ ]:
disp_triangle = cl.Triangle(
    disp_df,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Disposal Rate', 'Closed Claim Counts', 'Paid Claims'],
    cumulative=True,
)

dr_triangle = disp_triangle['Disposal Rate']
disp_closed_triangle = disp_triangle['Closed Claim Counts']
disp_paid_triangle = disp_triangle['Paid Claims']

disp_count_dev = cl.Development(average='volume', n_periods=-1).fit_transform(disp_closed_triangle)
disp_count_model = cl.Chainladder().fit(disp_count_dev)
ultimate_closed_counts = disp_count_model.ultimate_.to_frame().iloc[:, 0]

dr_long = triangle_to_frame(dr_triangle, origin_as_datetime=False).reset_index()
dr_matrix = dr_long.pivot(index='origin', columns='development', values='Disposal Rate').sort_index().sort_index(axis=1)
disp_closed_long = triangle_to_frame(disp_closed_triangle, origin_as_datetime=False).reset_index()
disp_closed_matrix = disp_closed_long.pivot(index='origin', columns='development', values='Closed Claim Counts').sort_index().sort_index(axis=1)
disp_paid_long = triangle_to_frame(disp_paid_triangle, origin_as_datetime=False).reset_index()
disp_paid_matrix = disp_paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)

age_cols = disp_closed_matrix.columns.astype(int)
selected_dr_by_age = pd.Series(index=age_cols, dtype=float)
for age in age_cols:
    vals = dr_matrix[age].dropna()
    selected_dr_by_age.loc[age] = vals.tail(5).mean() if len(vals) >= 5 else vals.mean()
selected_dr_by_age = selected_dr_by_age.ffill().clip(upper=1.0)
if pd.isna(selected_dr_by_age.iloc[-1]) or selected_dr_by_age.iloc[-1] < 0.999:
    selected_dr_by_age.iloc[-1] = 1.0

inc_paid_matrix = disp_paid_matrix.diff(axis=1)
inc_paid_matrix.iloc[:, 0] = disp_paid_matrix.iloc[:, 0]
inc_closed_matrix = disp_closed_matrix.diff(axis=1)
inc_closed_matrix.iloc[:, 0] = disp_closed_matrix.iloc[:, 0]
inc_severity_matrix = inc_paid_matrix / inc_closed_matrix

selected_inc_severity_by_age = pd.Series(index=age_cols, dtype=float)
for age in age_cols:
    vals = inc_severity_matrix[age].replace([np.inf, -np.inf], np.nan).dropna()
    selected_inc_severity_by_age.loc[age] = vals.tail(5).median() if len(vals) >= 5 else vals.median()
selected_inc_severity_by_age = selected_inc_severity_by_age.ffill()

tail_mask = selected_inc_severity_by_age.index >= 108
if tail_mask.any() and selected_inc_severity_by_age[tail_mask].notna().any():
    selected_tail_severity = float(selected_inc_severity_by_age[tail_mask].dropna().mean())
    selected_inc_severity_by_age.loc[tail_mask] = selected_tail_severity

selected_dr_by_age, selected_inc_severity_by_age


## Disposal Projection Results and Tail-Severity Considerations

Tail-severity selection notes:
- Late maturities often have sparse counts and unstable incremental severities.
- A common practice is to blend late-age severities into a selected tail severity.
- The impact should be judged relative to how much claim count remains open at those ages.


In [ ]:
latest_closed = disp_closed_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_paid_disp = disp_paid_triangle.latest_diagonal.to_frame().iloc[:, 0]

latest_age_disp = disp_closed_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_age_disp.index = pd.to_datetime(latest_age_disp.index.astype(str) + "-01-01")
latest_age_disp = latest_age_disp.reindex(ultimate_closed_counts.index)

projection_rows = []
for idx in ultimate_closed_counts.index:
    ay = idx.year
    latest_age_ay = int(latest_age_disp.loc[idx])
    u_closed = float(ultimate_closed_counts.loc[idx])
    latest_paid_ay = float(latest_paid_disp.loc[idx]) if pd.notna(latest_paid_disp.loc[idx]) else 0.0

    if latest_age_ay in selected_dr_by_age.index:
        dr_prev = float(selected_dr_by_age.loc[latest_age_ay])
    else:
        dr_prev = float(selected_dr_by_age[selected_dr_by_age.index <= latest_age_ay].iloc[-1])

    projected_future_unpaid = 0.0
    for age in age_cols:
        age = int(age)
        if age <= latest_age_ay:
            continue
        dr_age = float(selected_dr_by_age.loc[age])
        projected_incremental_closed = max(u_closed * (dr_age - dr_prev), 0.0)
        selected_sev_age = float(selected_inc_severity_by_age.loc[age])
        projected_future_unpaid += projected_incremental_closed * selected_sev_age
        dr_prev = dr_age

    projected_ultimate_paid = latest_paid_ay + projected_future_unpaid
    projection_rows.append(
        {
            'AccidentYear': ay,
            'LatestAge': latest_age_ay,
            'LatestClosedCounts': float(latest_closed.loc[idx]),
            'ProjectedUltimateClosedCounts': u_closed,
            'LatestPaidClaims': latest_paid_ay,
            'ProjectedUltimatePaidClaims': projected_ultimate_paid,
            'IBNR_DisposalMethod': projected_ultimate_paid - latest_paid_ay,
        }
    )

disposal_projection = pd.DataFrame(projection_rows).set_index('AccidentYear').sort_index()
disposal_projection.loc['Total'] = disposal_projection.sum()
disposal_projection


In [ ]:
final_summary = pd.DataFrame(
    {
        'Method': [
            'Freq-Sev Technique #1',
            'Freq-Sev Technique #2',
            'Disposal Rate Method',
        ],
        'TotalUltimateClaims': [
            float(ultimate_claims_t1.sum()),
            float(ultimate_claims_t2.sum()),
            float(disposal_projection.loc['Total', 'ProjectedUltimatePaidClaims']),
        ],
        'TotalIBNR': [
            float(ibnr_t1.sum()),
            float(ibnr_t2.sum()),
            float(disposal_projection.loc['Total', 'IBNR_DisposalMethod']),
        ],
    }
)

final_notes = pd.DataFrame(
    [
        ['Technique #1', 'Separates counts and severity development directly; sensitive to severity selection.'],
        ['Technique #2', 'Adds explicit exposure/frequency/severity trend structure; stronger assumption load.'],
        ['Disposal Rate', 'Useful when closed-count emergence is credible and incremental severity is stable enough by maturity.'],
    ],
    columns=['Method', 'Interpretation'],
)

final_summary, final_notes
